# Raw Momentum — 12M — Winner Drift

One signal, one formation horizon and one maintenance method. The experiment contains nine cells: rebalance every 1, 3 or 6 months × target N=12, 24 or 50.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
from momentum_india.notebook_views import ResearchNotebook

research = ResearchNotebook('raw_momentum', '12M', 'winner_drift')

## 1. Signal and portfolio rule

Score = adjusted close at the signal date / adjusted close at the formation start − 1. The start is the last common market close on or before the literal calendar-month offset. The latest month is included. Highest scores enter first.

Continuing holdings retain their naturally drifted weights, capped at 2/N at scheduled rebalances. Exits and entrants are paired in stable symbol order; an entrant receives min(exit weight, 1/N). Excess exit weight and cap trims are spread equally among continuing holdings with room below the cap. Unmatched entrants share existing cash up to 1/N each. Residual cash is retained. The cap is a scheduled target constraint, not a daily trim rule.

## 2. Universe → ranking → actual portfolio

The example uses the latest monthly N=24 signal and exposes the ranking inputs and actual target weights.

In [2]:
research.snapshot()

Symbol,Formation price return,MDTV (INR)
NSE:CUPID-EQ,664.2%,"2,261,224,535.95"
NSE:ATHERENERG-EQ,259.7%,"2,745,338,022.88"
NSE:KMEW-EQ,196.2%,"175,654,741.20"
NSE:RPTECH-EQ,190.7%,"78,147,051.55"
NSE:THANGAMAYL-EQ,176.4%,"572,585,716.40"
NSE:SILVERTUC-EQ,157.3%,"65,229,764.81"
NSE:HFCL-EQ,156.6%,"3,688,024,397.99"
NSE:SANSERA-EQ,150.7%,"511,549,310.60"
NSE:NGLFINE-EQ,148.1%,"21,228,409.00"
NSE:KIRLOSENG-EQ,142.2%,"594,920,683.95"


Symbol,Leg,Actual weight,Current selection,Execution status,Formation price return,MDTV (INR)
NSE:CUPID-EQ,long,3.0%,True,selected,664.2%,"2,261,224,535.95"
NSE:ATHERENERG-EQ,long,0.4%,True,selected,259.7%,"2,745,338,022.88"
NSE:KMEW-EQ,long,0.5%,True,selected,196.2%,"175,654,741.20"
NSE:RPTECH-EQ,long,0.4%,True,selected,190.7%,"78,147,051.55"
NSE:THANGAMAYL-EQ,long,0.8%,True,selected,176.4%,"572,585,716.40"
NSE:SILVERTUC-EQ,long,0.7%,True,selected,157.3%,"65,229,764.81"
NSE:HFCL-EQ,long,0.3%,True,selected,156.6%,"3,688,024,397.99"
NSE:SANSERA-EQ,long,0.3%,True,selected,150.7%,"511,549,310.60"
NSE:NGLFINE-EQ,long,0.3%,True,selected,148.1%,"21,228,409.00"
NSE:KIRLOSENG-EQ,long,0.8%,True,selected,142.2%,"594,920,683.95"


Symbol,Formation start,Start adjusted close,Signal close date,End adjusted close,Formation price return
NSE:CUPID-EQ,2025-07-31,30.18,2026-07-31,230.62,664.2%


## 3. Return layers across all nine cells

Raw is before trading charges. After-cost gross/pre-tax deducts modeled trading charges. The long-only post-tax overlay additionally applies the annual equity-gains ledger. The academic reference instead compares raw and borrowing-adjusted layers.

In [3]:
research.layers_bridge()

Rebalance,First date,Last date,Sessions
1M,2007-05-03,2026-08-28,4771
3M,2007-07-02,2026-08-28,4729
6M,2007-07-02,2026-08-28,4729


Rebalance,N,Raw,After costs,Post-tax overlay
1M,12,10.7%,10.3%,9.5%
1M,24,10.9%,10.4%,9.9%
1M,50,12.2%,11.8%,11.1%
3M,12,11.9%,11.5%,10.4%
3M,24,10.6%,10.2%,9.5%
3M,50,10.1%,9.7%,9.2%
6M,12,8.9%,8.7%,8.1%
6M,24,9.9%,9.6%,8.7%
6M,50,9.9%,9.6%,8.9%


## 4. Risk-adjusted results

Sharpe uses daily excess returns relative to the liquid fund. VaR and expected shortfall are historical monthly 95% loss measures. Partial first/last months are included. Time below prior peak counts days awaiting a new all-time high—not losing days. The initial invested capital is included as the first peak.

In [4]:
research.risk_grid()

Rebalance,N,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,12,9.5%,0.211,-75.1%,9.2%
1M,24,9.9%,0.240,-76.6%,7.4%
1M,50,11.1%,0.315,-71.1%,7.6%
3M,12,10.4%,0.249,-71.1%,10.3%
3M,24,9.5%,0.217,-74.9%,7.6%
3M,50,9.2%,0.201,-73.0%,8.0%
6M,12,8.1%,0.138,-66.1%,5.9%
6M,24,8.7%,0.171,-72.0%,9.5%
6M,50,8.9%,0.184,-74.6%,9.2%


Rebalance,N,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,12,16.2%,94.6%,1942
1M,24,14.4%,92.1%,1939
1M,50,14.0%,91.0%,1721
3M,12,16.0%,94.6%,1918
3M,24,14.6%,93.2%,1721
3M,50,14.9%,93.1%,1679
6M,12,14.2%,95.9%,2215
6M,24,16.0%,94.8%,1678
6M,50,16.6%,94.3%,1781


### CAGR

In [5]:
research.heatmap('cagr')

### Sharpe ratio

In [6]:
research.heatmap('sharpe')

### Maximum drawdown

In [7]:
research.heatmap('maximum_drawdown')

### Monthly 95% VaR

In [8]:
research.heatmap('monthly_var_95')

## 5. Equity paths and matched risks

Each chart fixes breadth and compares rebalance frequencies. Final-layer curves and the price benchmark start visible; other return layers remain in the selectable legend. Logarithmic axes make early and late periods comparable; the bottom range slider preserves the full history. Curves display weekly observations for readability, while every statistic uses the complete daily series.

### N=12

In [9]:
research.equity(12)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,10.7%,0.269,-74.0%,9.1%
1M,After costs,10.3%,0.248,-74.3%,9.2%
1M,Post-tax overlay,9.5%,0.211,-75.1%,9.2%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,11.9%,0.316,-70.1%,10.3%
3M,After costs,11.5%,0.299,-70.2%,10.3%
3M,Post-tax overlay,10.4%,0.249,-71.1%,10.3%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,8.9%,0.182,-64.7%,5.9%
6M,After costs,8.7%,0.171,-64.8%,5.9%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,15.8%,93.8%,1853
1M,After costs,16.0%,94.1%,1940
3M,Raw,15.7%,94.0%,1783
3M,After costs,15.8%,94.1%,1785
6M,Raw,13.9%,95.5%,2090
6M,After costs,14.0%,95.5%,2090
1M,Post-tax overlay,16.2%,94.6%,1942
3M,Post-tax overlay,16.0%,94.6%,1918
6M,Post-tax overlay,14.2%,95.9%,2215
1M,Nifty 50 price,13.1%,92.4%,1520


### N=24

In [10]:
research.equity(24)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,10.9%,0.298,-76.1%,7.4%
1M,After costs,10.4%,0.270,-76.3%,7.4%
1M,Post-tax overlay,9.9%,0.240,-76.6%,7.4%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,10.6%,0.278,-74.2%,7.5%
3M,After costs,10.2%,0.259,-74.3%,7.6%
3M,Post-tax overlay,9.5%,0.217,-74.9%,7.6%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,9.9%,0.232,-71.0%,9.4%
6M,After costs,9.6%,0.218,-71.1%,9.5%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,14.2%,90.8%,1782
1M,After costs,14.3%,91.4%,1935
3M,Raw,14.5%,91.9%,1653
3M,After costs,14.5%,92.2%,1685
6M,Raw,15.7%,94.1%,1644
6M,After costs,15.8%,94.2%,1647
1M,Post-tax overlay,14.4%,92.1%,1939
3M,Post-tax overlay,14.6%,93.2%,1721
6M,Post-tax overlay,16.0%,94.8%,1678
1M,Nifty 50 price,13.1%,92.4%,1520


### N=50

In [11]:
research.equity(50)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,12.2%,0.383,-70.4%,7.3%
1M,After costs,11.8%,0.357,-70.7%,7.6%
1M,Post-tax overlay,11.1%,0.315,-71.1%,7.6%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,10.1%,0.253,-71.9%,8.0%
3M,After costs,9.7%,0.230,-72.2%,8.0%
3M,Post-tax overlay,9.2%,0.201,-73.0%,8.0%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,9.9%,0.233,-73.6%,9.2%
6M,After costs,9.6%,0.219,-73.7%,9.2%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,13.8%,89.3%,1678
1M,After costs,13.9%,90.0%,1720
3M,Raw,14.7%,91.9%,1644
3M,After costs,14.7%,92.2%,1647
6M,Raw,16.3%,93.5%,1688
6M,After costs,16.4%,93.7%,1731
1M,Post-tax overlay,14.0%,91.0%,1721
3M,Post-tax overlay,14.9%,93.1%,1679
6M,Post-tax overlay,16.6%,94.3%,1781
1M,Nifty 50 price,13.1%,92.4%,1520


## 6. Recovery burden

The longest underwater episode is shown by its peak, trough and recovery dates. Unrecovered episodes remain explicitly open.

In [12]:
research.recovery()

Series,Peak,Trough,Recovery,Sessions,Calendar days,Episode loss
1M,2008-01-07,2008-12-05,2015-11-23,1939,2876,-76.6%
3M,2008-01-07,2008-12-03,2015-01-05,1721,2554,-74.9%
6M,2008-01-07,2009-03-09,2014-10-31,1678,2488,-72.0%
Nifty · 1M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%
Nifty · 3M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%
Nifty · 6M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%


## 7. Portfolio behavior

Scheduled turnover is (buy value + sell value)/(2 × pre-trade equity). Retention compares successive scheduled target name sets. Cash and total charges include the intervening daily path.

In [13]:
research.behavior()

Rebalance,N,Mean names at rebalance,Mean cash weight,Scheduled name retention
1M,12,16.34,27.6%,71.4%
1M,24,35.60,30.8%,76.6%
1M,50,72.47,27.2%,78.6%
3M,12,16.21,15.3%,51.0%
3M,24,35.38,25.7%,58.5%
3M,50,68.55,26.5%,60.5%
6M,12,16.51,30.2%,37.2%
6M,24,33.03,10.1%,40.5%
6M,50,67.21,10.3%,44.4%


Rebalance,N,Mean scheduled turnover,All trading charges (INR),Days with stop sales
1M,12,18.3%,"2,082,766.46",0
1M,24,13.3%,"1,582,733.45",0
1M,50,13.2%,"2,078,332.83",0
3M,12,38.5%,"1,797,066.01",0
3M,24,27.6%,"1,113,802.07",0
3M,50,26.3%,"1,078,606.68",0
6M,12,39.3%,"686,070.02",0
6M,24,51.6%,"1,016,422.49",0
6M,50,48.1%,"978,803.08",0


## 8. Market-state attribution

This is an observation, not an extra strategy filter. Prior-close Nifty 50 versus SMA(200) defines up/down; 63-session volatility versus its expanding historical median defines high/low volatility. The representative monthly N=24 path is shown with shaded states.

In [14]:
research.regimes()

Market state,Sessions,Mean daily return,Daily volatility,Positive days
Down / High volatility,748,-0.08%,1.27%,54.14%
Down / Low volatility,609,0.03%,0.89%,53.53%
Up / High volatility,720,0.14%,1.39%,61.53%
Up / Low volatility,2694,0.05%,0.79%,57.46%


## 9. Complete portfolio and trade evidence

Separate CSV files retain all scheduled portfolios, actual trades and risk layers for this exact signal/lookback/maintenance combination.

In [15]:
research.portfolio_exports()

Rebalance,N,First rebalance,Last rebalance,Rebalance dates,Holding rows
1M,12,2007-05-03,2026-08-03,232,3791
1M,24,2007-05-03,2026-08-03,232,8259
1M,50,2007-05-03,2026-08-03,232,16814
3M,12,2007-07-02,2026-07-01,77,1248
3M,24,2007-07-02,2026-07-01,77,2724
3M,50,2007-07-02,2026-07-01,77,5278
6M,12,2007-07-02,2026-07-01,39,644
6M,24,2007-07-02,2026-07-01,39,1288
6M,50,2007-07-02,2026-07-01,39,2621


## Findings

In [16]:
research.conclusion()